# ANR EEG Analysis Pipeline — Guided Google Colab

**African NeuroData Research Lab (ANR)**

Welcome to the guided ANR EEG analysis notebook. This notebook is designed so that a researcher or student can move through the workflow **one step at a time**, inspect each output, and understand what the pipeline is doing.

### What you will do
1. Install the current ANR EEG pipeline directly from GitHub.
2. Upload an EEG dataset.
3. Load the recording into MNE-Python.
4. Inspect channels, sampling rate, duration, and event markers.
5. Run technical quality control.
6. Choose preprocessing settings.
7. Preprocess the recording.
8. Inspect the power spectral density.
9. Calculate relative delta, theta, alpha, and beta power.
10. Review event markers.
11. Generate standardized ANR outputs and an HTML report.
12. Download the results.

> **Research use only:** The notebook produces technical and research-analysis outputs. It does not provide clinical EEG interpretation, diagnosis, or medical advice.

## Step 1 — Install the ANR EEG Pipeline from GitHub

The ANR GitHub repository is the single source of truth for this notebook. Running the cell below installs the current public version of the pipeline directly from the ANR Lab repository.

Run this cell whenever you start a fresh Colab session.

In [ ]:
import subprocess
import sys

REPO = "https://github.com/African-Neurodata-Research-Lab-ANR-LAB/ANR-EEG-Analysis-Pipeline.git"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}"
])

print("✓ ANR EEG Pipeline installed from GitHub")

## Step 2 — Import the analysis tools

This cell imports the ANR functions used throughout the notebook, together with MNE, pandas, matplotlib, and NumPy.

If this cell runs without an error, the installation is ready.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import anr_eeg

from anr_eeg import (
    load_eeg,
    run_qc,
    preprocess,
    compute_psd,
    compute_band_power,
    export_results,
    make_report,
)

print("✓ Imports successful")
print("ANR EEG version:", getattr(anr_eeg, "__version__", "unknown"))
print("MNE version:", mne.__version__)

## Step 3 — Upload your EEG dataset

Click **Choose Files** after running the cell below.

### Primary ANR format
The notebook is optimized for raw CSV files produced by the **ANR Muse EEG Recorder**. Those files normally contain:

- `TP9_uV`
- `AF7_uV`
- `AF8_uV`
- `TP10_uV`
- timestamps
- optional `event_marker`

### Other supported inputs
The Python package also supports common MNE-compatible EEG formats such as EDF/BDF, FIF, and BrainVision. BIDS datasets are supported through the Python API with an MNE-BIDS `BIDSPath`.

For this guided notebook, upload **one recording at a time** so you can inspect each stage carefully.

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No EEG file was uploaded.")

if len(uploaded) > 1:
    print("More than one file was uploaded. This notebook will use the first file only.")

file_path = next(iter(uploaded))
print("✓ Selected dataset:", file_path)

## Step 4 — Load the EEG recording

`load_eeg()` converts the uploaded dataset into an MNE `Raw` object.

For ANR Muse CSV recordings, the loader:
- checks the expected Muse EEG channels;
- converts microvolts to volts for MNE;
- estimates/validates the sampling rate;
- preserves ANR event markers as MNE annotations.

After running the cell, check that the channel names, sampling frequency, and recording duration are reasonable for your experiment.

In [ ]:
raw = load_eeg(file_path)

duration_s = raw.n_times / raw.info["sfreq"]

print(raw)
print()
print("Channels:", raw.ch_names)
print("Sampling frequency:", raw.info["sfreq"], "Hz")
print("Samples:", raw.n_times)
print("Duration:", round(duration_s, 2), "seconds")
print("Annotations/events:", len(raw.annotations))

## Step 5 — Inspect a short segment of the raw EEG

This visualization is for checking the acquired waveform before preprocessing.

The plot does **not** diagnose abnormal brain activity. Look primarily for gross recording problems such as flat channels, very large movement artifacts, or obvious discontinuities.

You can change `PREVIEW_SECONDS` if you want a longer or shorter preview.

In [ ]:
PREVIEW_SECONDS = min(10.0, duration_s)

raw.plot(
    duration=PREVIEW_SECONDS,
    n_channels=min(4, len(raw.ch_names)),
    scalings="auto",
    show=False,
)

plt.show()

## Step 6 — Run technical acquisition quality control

ANR QC summarizes properties of the acquired signal such as:
- channel count;
- recording duration;
- sampling frequency;
- per-channel standard deviation;
- peak-to-peak amplitude;
- flat-channel/high-amplitude flags;
- number of event annotations.

Possible overall statuses are `pass`, `review`, or `fail`.

These labels describe **technical acquisition quality only**.

In [ ]:
qc = run_qc(raw)

print("Overall technical QC status:", qc["status"].upper())
print("Sampling frequency:", qc["sampling_frequency_hz"], "Hz")
print("Duration:", round(qc["duration_seconds"], 2), "seconds")
print("Channel count:", qc["channel_count"])
print("Annotations:", qc["annotation_count"])

qc_table = pd.DataFrame(qc["channels"]).T
display(qc_table)

## Step 7 — Choose preprocessing settings

Before running preprocessing, review the settings below.

### Recommended starting values for many Muse research recordings
- **Low cut:** 1 Hz
- **High cut:** 40 Hz
- **Notch:** 50 Hz in countries using 50 Hz mains electricity
- **Reference:** `None` keeps the existing reference

Do not choose filter settings only because they are defaults. Your experimental design and downstream analysis should determine the final parameters.

If you work in a 60 Hz mains region, change `NOTCH_HZ` to `60.0`.

In [ ]:
LOW_CUT_HZ = 1.0
HIGH_CUT_HZ = 40.0
NOTCH_HZ = 50.0
REFERENCE = None

print("Preprocessing settings")
print("----------------------")
print("Band-pass:", LOW_CUT_HZ, "to", HIGH_CUT_HZ, "Hz")
print("Notch:", NOTCH_HZ, "Hz")
print("Reference:", REFERENCE)

## Step 8 — Preprocess the EEG

The ANR preprocessing function works on a **copy** of the raw MNE object.

The v1 workflow performs:
1. optional notch filtering;
2. band-pass filtering;
3. optional EEG re-referencing.

The original `raw` variable remains unchanged, while the processed recording is stored as `clean`.

In [ ]:
clean = preprocess(
    raw,
    l_freq=LOW_CUT_HZ,
    h_freq=HIGH_CUT_HZ,
    notch=NOTCH_HZ,
    reference=REFERENCE,
)

print("✓ Preprocessing complete")
print(clean)

## Step 9 — Compare a short segment after preprocessing

Use this cell to visually inspect the processed signal.

Filtering changes the frequency content of the recording; it should not be treated as a way to make every waveform look visually smooth.

In [ ]:
clean.plot(
    duration=PREVIEW_SECONDS,
    n_channels=min(4, len(clean.ch_names)),
    scalings="auto",
    show=False,
)

plt.show()

## Step 10 — Power spectral density (PSD)

The power spectral density describes how signal power is distributed across frequencies.

The ANR v1 pipeline uses MNE's Welch PSD implementation. For the default analysis, the spectrum is evaluated from 1–40 Hz.

When reading the plot, consider your experimental condition, preprocessing settings, recording length, and artifacts. A spectral peak alone is not a diagnosis.

In [ ]:
spectrum = compute_psd(clean, fmin=1.0, fmax=40.0)

spectrum.plot(show=False)
plt.show()

## Step 11 — Relative EEG band power

The v1 pipeline calculates four standard frequency ranges:

| Band | Frequency |
|---|---:|
| Delta | 1–4 Hz |
| Theta | 4–8 Hz |
| Alpha | 8–13 Hz |
| Beta | 13–30 Hz |

Band values are expressed **relative to total 1–30 Hz power for each channel**.

This makes the table useful for comparisons across channels or experimental conditions, but it is not a clinical classification.

In [ ]:
bands = compute_band_power(clean)

band_percent = (bands * 100).round(2)
band_percent.columns = [f"{column}_pct" for column in band_percent.columns]

display(band_percent)

## Step 12 — Visualize relative band power

The plot below provides an easy visual comparison of frequency-band composition across EEG channels.

In [ ]:
ax = band_percent.plot(kind="bar", figsize=(10, 5))
ax.set_title("ANR EEG Relative Band Power")
ax.set_xlabel("EEG channel")
ax.set_ylabel("Relative power (%)")
ax.legend(title="Frequency band")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 13 — Review event markers

If your recording came from the ANR Muse EEG Recorder and contains event markers, they were converted into MNE annotations during import.

Review the event table before interpreting task-related changes. Correct event timing is essential for later ERP and time-frequency workflows.

In [ ]:
if len(clean.annotations) == 0:
    print("No event annotations were found in this recording.")
else:
    events_table = pd.DataFrame({
        "onset_seconds": clean.annotations.onset,
        "duration_seconds": clean.annotations.duration,
        "event": clean.annotations.description,
    })
    display(events_table)

## Step 14 — Generate standardized ANR research outputs

The export step creates reusable files for later analysis and documentation.

The v1 pipeline generates:
- cleaned EEG in MNE FIF format;
- technical QC JSON;
- relative band-power CSV;
- analysis summary JSON;
- MNE-based HTML report.

Change `SESSION_PREFIX` to a non-identifying research/session code before exporting.

In [ ]:
from pathlib import Path

SESSION_PREFIX = "anr_session"
output_dir = Path("/content/anr_eeg_results")

outputs = export_results(
    clean,
    qc,
    bands,
    output_dir,
    prefix=SESSION_PREFIX,
)

report_path = make_report(
    clean,
    qc,
    bands,
    output_dir / f"{SESSION_PREFIX}_report.html",
)

print("Generated outputs")
print("-----------------")
for name, path in outputs.items():
    print(f"{name}: {path}")

print("html_report:", report_path)

## Step 15 — Inspect the generated files

Before downloading, confirm that all expected outputs exist.

You may also open the HTML report from Colab's file browser to inspect it before downloading the complete result package.

In [ ]:
generated_files = sorted(output_dir.glob("*"))

for path in generated_files:
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:45s} {size_kb:10.1f} KB")

## Step 16 — Download the complete ANR result package

Run this cell only when you are satisfied with the workflow above.

It compresses the generated result folder into one ZIP file and downloads it to your computer.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "/content/ANR_EEG_results",
    "zip",
    output_dir,
)

print("Downloading:", archive_path)
files.download(archive_path)

# Finished

You have completed the ANR EEG v1 guided workflow.

### Your analysis path
**Dataset → MNE import → technical QC → preprocessing → PSD → band power → event review → standardized exports → HTML report**

### What comes next?
The validated v1 core intentionally stops here. Future ANR modules may add:
- epoching and ERP;
- time-frequency analysis;
- ICA/artifact workflows;
- connectivity;
- machine-learning/decoding workflows.

Those modules should be used only when the research design and data support them.

For repository documentation and navigation, read:
- `docs/GETTING_STARTED.md`
- `docs/REPOSITORY_GUIDE.md`
- `docs/OUTPUTS_AND_INTERPRETATION.md`